## 1. Importing and Loading Data

In [68]:
import sys
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yaml
from pprint import pprint
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder, OrdinalEncoder, TargetEncoder, StandardScaler, MinMaxScaler, RobustScaler,
)
from category_encoders import CountEncoder
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import SelectKBest, f_classif
from scipy.stats import chi2_contingency


from src.cleaning import clean_data

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
BLUE, ORANGE, GRAY = "#3B9BDB", "#f35b1f", "#A0A9BB"
mpl.rcParams["figure.dpi"] = 110
mpl.rcParams["axes.spines.top"] = False
mpl.rcParams["axes.spines.right"] = False
mpl.rcParams["font.size"] = 10.5

In [69]:
# Find the project root
ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Add the project root to the Python path
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

Project root: C:\Users\Bea\Documents\NOVA IMS\Masters DSAA\Fall\ETAI\Project\ETAI-Final-Project-G25-main\ETAI-Final-Project-G25-main


## 2. Cleaning and Feature/Target Separation

In [70]:
train = pd.read_csv(ROOT / "data" / "raw" / "train.csv")
test = pd.read_csv(ROOT / "data" / "raw" / "test.csv")

print("Train:", train.shape)
print("Test:", test.shape)

Train: (6534, 40)
Test: (986, 38)


In [71]:
# Apply cleaning to copies of the original datasets
train_clean = clean_data(train)
test_clean = clean_data(test)

print("Original train:", train.shape)
print("Clean train:", train_clean.shape)

print("Original test:", test.shape)
print("Clean test:", test_clean.shape)

Original train: (6534, 40)
Clean train: (6534, 40)
Original test: (986, 38)
Clean test: (986, 38)


In [72]:
# Define targets
classification_target = "repeat_purchase_90d"
regression_target = "spend_next_90d"

# Classification and regression targets
y_class = train_clean[classification_target].copy()
y_reg = train_clean[regression_target].copy()

# Predictor variables
X = train_clean.drop(columns=[
    classification_target,
    regression_target,
    "ID",
    "customer_id",
    "purchase_date"
])

print("Features:", X.shape)
print("Classification target:", y_class.shape)
print("Regression target:", y_reg.shape)

Features: (6534, 35)
Classification target: (6534,)
Regression target: (6534,)


In [73]:
# Metric features
metric_features = [
    'Total a Pagar', 'Total Bruto', 'Total Liquido',
    'Total Imp_', 'Total Portes', 'Total Desc_ Global',
    'Total Desc_ Linha', 'Custo', 'N_ Detalhes',
    'invoice_count_current_purchase',
    'days_since_previous_purchase', 'customer_tenure_days',
    'prior_purchase_count', 'prior_total_spend',
    'prior_mean_purchase_value', 'prior_std_purchase_value',
    'prior_min_purchase_value', 'prior_max_purchase_value',
    'prior_mean_gap_days', 'prior_std_gap_days',
    'purchase_count_30d', 'spend_30d',
    'purchase_count_90d', 'spend_90d',
    'purchase_count_180d', 'spend_180d',
    'purchase_count_365d', 'spend_365d',
    'random_noise'
]

# Non-metric features
non_metric_features = [
    'payment_type', 'salesperson', 'commercial_zone',
    'transport_method', 'warehouse', 'document_series'
]

# Datetime features
datetime_features = ['purchase_date']

# Unique identifiers
id_features = ['ID', 'customer_id']

# Target variables
target_features = ['repeat_purchase_90d', 'spend_next_90d']

In [74]:
train_clean[metric_features].describe().T

,count,mean,std,min,25%,50%,75%,max
Total a Pagar,6469.0,9.255563e+02,1.334998e+03,0.93000,188.920000,476.740000,1087.670000,2.229721e+04
Total Bruto,6469.0,8.861527e+02,1.327641e+03,0.83000,175.740000,445.740000,1036.700000,2.157828e+04
Total Liquido,6469.0,7.733364e+02,1.136356e+03,0.76000,154.380000,389.910000,903.370000,1.812781e+04
Total Imp_,6469.0,1.522199e+02,2.252624e+02,0.00000,29.930000,79.740000,184.840000,4.169400e+03
Total Portes,6469.0,1.151427e+00,5.844091e+00,0.00000,0.000000,0.000000,0.000000,1.810600e+02
Total Desc_ Global,6469.0,3.499072e-01,5.116564e+00,0.00000,0.000000,0.000000,0.000000,2.184900e+02
Total Desc_ Linha,6469.0,1.124664e+02,2.899863e+02,0.00000,0.000000,13.130000,92.280000,4.103440e+03
Custo,6469.0,3.281650e+02,5.195072e+02,0.00000,49.050000,152.540000,396.061499,7.427401e+03
N_ Detalhes,6469.0,2.152450e+01,2.771342e+01,1.00000,4.000000,11.000000,29.000000,4.230000e+02
invoice_count_current_purchase,6469.0,1.066007e+00,3.108069e-01,1.00000,1.000000,1.000000,1.000000,5.000000e+00


## 3. Temporal Train/Validation Split

In [75]:
# Chronological holdout with a 90-day label-availability gap.
dated = train_clean.loc[train_clean["purchase_date"].notna()].sort_values("purchase_date")
assert len(dated) > 0, "No parseable purchase_date values."

split_position = int(len(dated) * 0.80)
validation_start = dated["purchase_date"].iloc[split_position]

train_mask = dated["purchase_date"] + pd.Timedelta(days=90) <= validation_start
val_mask = dated["purchase_date"] >= validation_start

train_idx = dated.index[train_mask]
val_idx = dated.index[val_mask]

assert len(train_idx) and len(val_idx), "Temporal split produced an empty partition."
assert train_idx.intersection(val_idx).empty

X_train, X_val = X.loc[train_idx].copy(), X.loc[val_idx].copy()
y_train, y_val = y_class.loc[train_idx].copy(), y_class.loc[val_idx].copy()

print("Validation begins:", validation_start.date())
print("Classification train:", X_train.shape, "validation:", X_val.shape)
print("Rows excluded because dates are missing:", train_clean["purchase_date"].isna().sum())
print("Rows in the 90-day gap:", len(dated) - len(train_idx) - len(val_idx))
print("Positive rate — train:", round(y_train.mean(), 3), "validation:", round(y_val.mean(), 3))

Validation begins: 2023-06-17
Classification train: (5024, 35) validation: (1298, 35)
Rows excluded because dates are missing: 65
Rows in the 90-day gap: 147
Positive rate — train: 0.55 validation: 0.603


In [76]:
# Regression is conditional on repeat purchase and an observed spend target.
reg_train_idx = train_idx[y_class.loc[train_idx].eq(1) & y_reg.loc[train_idx].notna()]
reg_val_idx = val_idx[y_class.loc[val_idx].eq(1) & y_reg.loc[val_idx].notna()]
X_train_reg, X_val_reg = X.loc[reg_train_idx].copy(), X.loc[reg_val_idx].copy()
y_train_reg, y_val_reg = y_reg.loc[reg_train_idx].copy(), y_reg.loc[reg_val_idx].copy()

print("Regression train:", X_train_reg.shape, "validation:", X_val_reg.shape)
print("Returning train rows with missing regression target:", int((y_class.loc[train_idx].eq(1) & y_reg.loc[train_idx].isna()).sum()))

Regression train: (2656, 35) validation: (694, 35)
Returning train rows with missing regression target: 106


## 4. Missing value Imputation

### 4.1. Non-metric Features

In [77]:
# Work with copies to preserve the original data
train_imp = X_train.copy()
val_imp = X_val.copy()

# Variable to impute
target_col = "payment_type"

print("Missing before imputation:", train_imp[target_col].isna().sum())

# Separate observations with known and missing values
known = train_imp[train_imp[target_col].notna()].copy()
unknown = train_imp[train_imp[target_col].isna()].copy()

# Use all available predictors except the variable being imputed
features = [col for col in X_train.columns if col != target_col]

# Separate numerical and categorical predictors
num_features = [col for col in metric_features if col in features]
cat_features = [col for col in non_metric_features if col in features]

# Prepare numerical and categorical predictors
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_features)
])

# Encode the target categories
le = LabelEncoder()
known["target_encoded"] = le.fit_transform(known[target_col])

# Separate features and target
X_known = known[features]
y_known = known["target_encoded"]

# Define the Decision Tree
clf = Pipeline([
    ("preprocessor", preprocessor),
    ("tree", DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ))
])

# Train using only the training set
clf.fit(X_known, y_known)

# Predict missing values in training
if len(unknown) > 0:
    y_pred = clf.predict(unknown[features])
    train_imp.loc[unknown.index, target_col] = le.inverse_transform(y_pred)

# Predict missing values in validation
missing_val = val_imp[target_col].isna()

if missing_val.any():
    y_pred_val = clf.predict(val_imp.loc[missing_val, features])
    val_imp.loc[missing_val, target_col] = le.inverse_transform(y_pred_val)

print("Missing after imputation:", train_imp[target_col].isna().sum())
print("Validation missing after:", val_imp[target_col].isna().sum())


Missing before imputation: 144
Missing after imputation: 0
Validation missing after: 0


In [78]:
# Remaining categorical variables to impute
remaining_categorical = [
    "salesperson",
    "commercial_zone",
    "transport_method",
    "warehouse",
    "document_series"
]

for target_col in remaining_categorical:

    print(f"\nImputing: {target_col}")
    print("Missing before:", train_imp[target_col].isna().sum())

    # Separate observations where the category is known and unknown
    known = X_train[X_train[target_col].notna()].copy()
    unknown = train_imp[train_imp[target_col].isna()].copy()

    # Use all other available predictors
    features = [col for col in X_train.columns if col != target_col]

    # Separate numerical and categorical predictors
    num_features = [col for col in metric_features if col in features]
    cat_features = [col for col in non_metric_features if col in features]

    # Encode the target variable
    le = LabelEncoder()
    known["target_encoded"] = le.fit_transform(known[target_col])

    X_known = known[features]
    y_known = known["target_encoded"]

    # Preprocess predictors before training the tree
    preprocessor = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_features)
    ])

    # Train a separate decision tree for each categorical variable
    clf = Pipeline([
        ("preprocessor", preprocessor),
        ("tree", DecisionTreeClassifier(
            max_depth=5,
            random_state=42
        ))
    ])

    clf.fit(X_known, y_known)

    # Predict missing categories in training data
    if len(unknown) > 0:
        X_unknown = unknown[features]
        y_pred = clf.predict(X_unknown)

        train_imp.loc[unknown.index, target_col] = le.inverse_transform(y_pred)

    # Predict missing categories in validation data
    missing_val = val_imp[target_col].isna()

    if missing_val.any():
        X_unknown_val = val_imp.loc[missing_val, features]
        y_pred_val = clf.predict(X_unknown_val)

        val_imp.loc[missing_val, target_col] = le.inverse_transform(y_pred_val)

    print("Missing after:", train_imp[target_col].isna().sum())
    print("Validation missing after:", val_imp[target_col].isna().sum())


Imputing: salesperson
Missing before: 158
Missing after: 0
Validation missing after: 0

Imputing: commercial_zone
Missing before: 0
Missing after: 0
Validation missing after: 0

Imputing: transport_method
Missing before: 153
Missing after: 0
Validation missing after: 0

Imputing: warehouse
Missing before: 150
Missing after: 0
Validation missing after: 0

Imputing: document_series
Missing before: 0
Missing after: 0
Validation missing after: 0


In [79]:
# Categorical distributions after imputation
for col in non_metric_features:
    print(f"\n--- {col} ---")
    print(train_imp[col].value_counts(dropna=False).to_string())


--- payment_type ---
payment_type
Maybe Tomorrow Payment 7     3709
Maybe Tomorrow Payment 12     780
Maybe Tomorrow Payment 9      415
Maybe Tomorrow Payment 10      88
Maybe Tomorrow Payment 8       25
Maybe Tomorrow Payment 11       6
Maybe Tomorrow Payment 3        1

--- salesperson ---
salesperson
VLOOKUP Master 1    3420
VLOOKUP Master 4    1131
VLOOKUP Master 7     202
VLOOKUP Master 2     128
VLOOKUP Master 9      94
VLOOKUP Master 3      30
VLOOKUP Master 6      12
VLOOKUP Master 5       4
VLOOKUP Master 8       3

--- commercial_zone ---
commercial_zone
Decorative Dashboard Zone 4    4901
Decorative Dashboard Zone 2      78
Decorative Dashboard Zone 3      43
Decorative Dashboard Zone 6       1
Decorative Dashboard Zone 7       1

--- transport_method ---
transport_method
Maybe Tomorrow Delivery 72    1025
Maybe Tomorrow Delivery 59     356
Maybe Tomorrow Delivery 61     314
Maybe Tomorrow Delivery 73     297
Maybe Tomorrow Delivery 11     264
Maybe Tomorrow Delivery 40    

### 4.2. Metric Features

NAO ESTA A FUNCIONAR BEMMMMM!!

In [80]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Work with copies to preserve the original datasets
train_mice = train_imp.copy()
val_mice = val_imp.copy()

# Numerical features
features = [col for col in metric_features if col in features]

# Check missing values before imputation
print("Missing before:", train_mice[num_features].isna().sum().sum())

# Define MICE imputer
mice = IterativeImputer(
    max_iter=10,
    random_state=42
)

# Fit MICE only on training data
train_mice[features] = mice.fit_transform(train_mice[features])

# Apply the same imputer to validation data
val_mice[features] = mice.transform(val_mice[features])

# Check missing values after imputation
print("Missing after:", train_mice[features].isna().sum().sum())
print("Validation missing after:", val_mice[features].isna().sum().sum())

Missing before: 6062
Missing after: 0
Validation missing after: 0


In [81]:
# Check missing values after MICE
train_mice.isna().sum()

Total a Pagar                     0
Total Bruto                       0
Total Liquido                     0
Total Imp_                        0
Total Portes                      0
Total Desc_ Global                0
Total Desc_ Linha                 0
Custo                             0
N_ Detalhes                       0
invoice_count_current_purchase    0
days_since_previous_purchase      0
customer_tenure_days              0
prior_purchase_count              0
prior_total_spend                 0
prior_mean_purchase_value         0
prior_std_purchase_value          0
prior_min_purchase_value          0
prior_max_purchase_value          0
prior_mean_gap_days               0
prior_std_gap_days                0
purchase_count_30d                0
spend_30d                         0
purchase_count_90d                0
spend_90d                         0
purchase_count_180d               0
spend_180d                        0
purchase_count_365d               0
spend_365d                  

In [82]:
# General statistics of the metric features
train_mice[metric_features].describe().T

,count,mean,std,min,25%,50%,75%,max
Total a Pagar,5024.0,8.377821e+02,1.174711e+03,0.930000,172.835000,435.600000,1001.740000,1.443341e+04
Total Bruto,5024.0,7.909183e+02,1.157601e+03,0.830000,157.387500,404.635000,928.670000,1.344346e+04
Total Liquido,5024.0,6.989271e+02,1.004933e+03,0.760000,141.150000,357.230000,825.012500,1.173448e+04
Total Imp_,5024.0,1.388550e+02,1.929536e+02,0.000000,27.705000,74.555000,171.395000,2.698930e+03
Total Portes,5024.0,1.228465e+00,6.104400e+00,0.000000,0.000000,0.000000,0.000000,1.810600e+02
Total Desc_ Global,5024.0,3.070541e-01,5.140932e+00,0.000000,0.000000,0.000000,0.000000,2.184900e+02
Total Desc_ Linha,5024.0,9.168416e+01,2.534793e+02,0.000000,0.000000,7.455000,71.975000,3.794510e+03
Custo,5024.0,2.963569e+02,4.520334e+02,0.000000,43.730000,141.038285,360.846419,6.111300e+03
N_ Detalhes,5024.0,2.055314e+01,2.730347e+01,1.000000,3.000000,10.000000,27.000000,4.230000e+02
invoice_count_current_purchase,5024.0,1.069865e+00,3.206595e-01,1.000000,1.000000,1.000000,1.000000,5.000000e+00


## 5. Encoding

AINDA POR FFAZEERRRRRRRR

## 6. Feature Selection

### 6.1. Chi-squared Test

The Chi-Squared Test for Independence checks if there's a significant relationship between a categorical feature and the target variable. If they’re dependent, the feature might be useful for prediction.

In [ ]:
def TestIndependence(X, y, var, alpha=0.05):
    """Test association between a categorical predictor and the target."""

    dfObserved = pd.crosstab(y, X)
    chi2, p, dof, expected = chi2_contingency(dfObserved.values)

    dfExpected = pd.DataFrame(
        expected,
        columns=dfObserved.columns,
        index=dfObserved.index
    )

    if p < alpha:
        result = "{0} is associated with the target".format(var)
    else:
        result = "No significant association found for {0}".format(var)

    print(result)

In [59]:
# Categorical features
categorical_features = [
    "payment_type",
    "salesperson",
    "commercial_zone",
    "transport_method",
    "warehouse",
    "document_series"
]

# Test association with the classification target
for var in categorical_features:
    TestIndependence(
        X_train[var].fillna("Missing"),
        y_train,
        var
    )

payment_type is associated with the target
salesperson is associated with the target
commercial_zone is associated with the target
transport_method is associated with the target
warehouse is associated with the target
document_series is associated with the target


### 6.2. ANOVA Test

The ANOVA F-test is used to evaluate whether numerical features have
significantly different means between customers who repeat a purchase
and those who do not.

The test is fitted only on training data to avoid data leakage.

In [ ]:
# ANOVA on numerical features after MICE imputation
anova = SelectKBest(f_classif, k="all")

X_anova = anova.fit_transform(
    train_mice[metric_features],
    y_train
)

# ANOVA p-values
p_values = pd.Series(
    anova.pvalues_,
    index=metric_features
)

In [67]:
#Features with p-value above 0.05
selected_features

Total a Pagar                      True
Total Bruto                        True
Total Liquido                      True
Total Imp_                         True
Total Portes                      False
Total Desc_ Global                False
Total Desc_ Linha                  True
Custo                              True
N_ Detalhes                       False
invoice_count_current_purchase     True
days_since_previous_purchase       True
customer_tenure_days               True
prior_purchase_count               True
prior_total_spend                  True
prior_mean_purchase_value         False
prior_std_purchase_value           True
prior_min_purchase_value           True
prior_max_purchase_value           True
prior_mean_gap_days                True
prior_std_gap_days                 True
purchase_count_30d                 True
spend_30d                          True
purchase_count_90d                 True
spend_90d                          True
purchase_count_180d                True


In [62]:
p_values

Total a Pagar                      7.218550e-06
Total Bruto                        2.322573e-06
Total Liquido                      3.217047e-05
Total Imp_                         1.468929e-08
Total Portes                       4.014376e-01
Total Desc_ Global                 4.718028e-01
Total Desc_ Linha                  3.425303e-07
Custo                              1.047066e-02
N_ Detalhes                        7.029551e-01
invoice_count_current_purchase     1.401733e-04
days_since_previous_purchase       1.798159e-48
customer_tenure_days               1.568133e-08
prior_purchase_count               7.119751e-96
prior_total_spend                  8.971606e-84
prior_mean_purchase_value          5.272280e-01
prior_std_purchase_value           4.224366e-20
prior_min_purchase_value           1.158706e-13
prior_max_purchase_value           2.460312e-12
prior_mean_gap_days                2.022423e-74
prior_std_gap_days                 5.616301e-37
purchase_count_30d                 1.391